# Ch 25. 양자화 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: INT8 양자화 후 선형층 출력의 MSE를 측정하라.

### 풀이

대칭 INT8에서는 $s=\max|W|/127$, $\hat W=s\,\mathrm{clip}(\mathrm{round}(W/s),-127,127)$이다. 기준 출력 $Y=XW^T$와 $\hat Y=X\hat W^T$에 대해 $\mathrm{MSE}=|Y-\hat Y|_F^2/n$을 계산한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(25); x=rng.normal(size=(64,32)); w=rng.normal(size=(16,32)); y=x@w.T; yq=x@quantize(w).T
mse=np.mean((y-yq)**2); assert mse>0 and np.isfinite(mse); mse


## 문제 2

**문제**: Per-tensor vs Per-channel vs Per-group 양자화 오차를 비교하라.

### 풀이

스케일 하나를 공유하는 범위가 작을수록 이상치의 영향이 국소화된다. Per-channel은 출력 행마다, per-group은 연속 열 그룹마다 스케일을 추정하고 같은 가중 MSE로 비교한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(2); w=rng.normal(size=(8,64))*np.arange(1,9)[:,None]
pt=np.mean((w-quantize(w))**2); pc=np.mean((w-quantize(w,axis=1))**2)
pg=np.mean(np.concatenate([w[:,i:i+16]-quantize(w[:,i:i+16]) for i in range(0,64,16)],1)**2)
assert pc<=pt and pg<=pt; {'tensor':pt,'channel':pc,'group':pg}


## 문제 3

**문제**: 그룹 크기 32, 64, 128, 256에 따른 양자화 오차를 비교하라.

### 풀이

그룹 크기 $g$가 작으면 스케일 수는 늘고 양자화 오차는 보통 줄지만 메타데이터와 커널 비용이 증가한다. 동일 행렬을 정확히 나눈 뒤 전체 원소 MSE를 계산해야 한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import numpy as np
def quantize(x, axis=None):
 s=np.maximum(np.max(np.abs(x),axis=axis,keepdims=True)/127,1e-12); q=np.clip(np.round(x/s),-127,127); return q*s
rng=np.random.default_rng(3); w=rng.normal(size=(4,256))*np.linspace(.2,3,256)
errs={g:np.mean((w-np.concatenate([quantize(w[:,i:i+g]) for i in range(0,256,g)],1))**2) for g in [32,64,128,256]}
assert errs[32] <= errs[256]; errs


## 문제 4

**문제**: FP32 vs FP16 추론 속도를 CPU와 GPU에서 각각 비교하라.

### 풀이

속도는 dtype만으로 결정되지 않는다. CPU/GPU별로 같은 shape와 연산을 사용하고 워밍업 뒤 GPU 동기화를 포함한 반복 중앙값을 측정하며, 해당 장치가 FP16을 가속하는지도 보고한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import time, numpy as np
rng=np.random.default_rng(254); x=rng.normal(size=(128,256)); w=rng.normal(size=(256,256))
scale=max(float(np.abs(w).max())/127,1e-12); q=np.clip(np.rint(w/scale),-127,127).astype(np.int8); wq=q.astype(np.float64)*scale

def median_runtime(fn, repetitions=31):
    samples=[]
    for _ in range(repetitions):
        start=time.perf_counter(); value=fn(); samples.append(time.perf_counter()-start)
    return float(np.median(samples)), value

fp_time, fp_out=median_runtime(lambda: x@w)
quant_time, quant_out=median_runtime(lambda: x@wq)
reference=x@(q.astype(np.float64)*scale)
assert np.allclose(quant_out, reference) and np.array_equal(fp_out, x@w)
assert fp_time>0 and quant_time>0 and np.isfinite(np.max(np.abs(fp_out-quant_out)))
{'repetitions':31,'fp_median_seconds':fp_time,'dequantized_int8_median_seconds':quant_time,
 'max_abs_output_error':float(np.max(np.abs(fp_out-quant_out)))}


## 문제 5

**문제**: QLoRA가 풀 파인튜닝과 비교해 메모리를 얼마나 절약하는지 7B, 13B, 70B 모델에 대해 계산하라.

### 풀이

Adam 풀 FT의 모델·그래디언트·master weight·모멘트 근사는 파라미터당 16 byte이다. QLoRA는 4-bit base 약 0.5 byte와 작은 LoRA/optimizer 상태만 학습하므로 코드의 명시적 가정 아래 절감량을 계산한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
models=[7,13,70]; full_bytes=16; qlora_base=.5; adapter_fraction=.01; adapter_bytes=16
rows=[(p,p*full_bytes,p*(qlora_base+adapter_fraction*adapter_bytes)) for p in models]
assert all(q<f for _,f,q in rows); rows
